# 뉴스 요약하고 문서화하기

- pip install --upgrade openai 

### API key 로딩

In [2]:
from dotenv import load_dotenv
import os

# .env 파일 로드, # 기존 환경변수 덮어쓰기
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

### 라이브러리 로딩 및 개체 생성

In [3]:
from openai import OpenAI
# openai api인증 및 OpenAI객체생성
client = OpenAI(api_key = OPENAI_API_KEY)

- 일단 변수에 프롬포트 입력해두기

In [4]:
summary_prompt = """당신은 언어 이해 및 요약에 훈련된 고도로 숙련된 AI입니다.
다음 텍스트를 읽고 간결한 추상적 단락으로 요약했으면 합니다.
전체 텍스트를 읽을 필요 없이 토론의 요점을 이해하는 데 도움이 될 수 있는 일관되고
읽을 수 있는 요약을 제공하여 가장 중요한 요점을 유지하는 것을 목표로 합니다.
불필요한 세부 사항이나 접선 사항은 피하십시오."""
key_points_prompt = """당신은 정보를 핵심 포인트로 전달하는 데 특화된 능숙한 AI입니다.
다음 텍스트를 기반으로 논의되거나 언급된 주요 포인트를 확인하고 나열합니다.
이는 논의의 본질에 가장 중요한 아이디어, 결과 또는 주제가 되어야 합니다.
당신의 목표는 누군가가 읽을 수 있는 목록을 제공하여 이야기된 내용을 빠르게 이해하는 것입니다."""
action_items_prompt = """당신은 대화를 분석하고 행동 항목을 추출하는 데 있어 AI 전문가입니다.
본문을 검토하고 합의되거나 수행이 필요하다고 언급된 모든 작업, 과제 또는 행동을 식별하십시오.
이것들은 특정 개인에게 할당된 작업일 수도 있고 그룹이 취하기로 결정한 일반적인 행동일 수도 있습니다.
이러한 행동 항목을 명확하고 간결하게 나열하십시오."""
sentiment_prompt = """당신은 언어와 감정 분석에 전문성을 갖춘 AI로서 당신의 과제는 다음 텍스트의 감
정을 분석하는 것입니다.
토론의 전체적인 톤, 사용된 언어가 전달하는 감정, 단어와 구가 사용되는 맥락을 고려하십시오.
감정이 일반적으로 긍정적인지 부정적인지 중립적인지를 표시하고 가능한 한 당신의 분석에 대해 간략한 설명을 제
공하십시오."""

### 번역 함수

In [5]:
def text_extraction(text, prompt):
    response = client.chat.completions.create(
        model="gpt-4-turbo",
        temperature=0,
        messages=[
        { "role": "system", "content": prompt },
        { "role": "user", "content": text }
        ]
    )
    return response.choices[0].message.content

### 음성파일 > txt로 전환 / 넘어감

In [6]:
audio_file_path = "datas/news.mp3"
with open(audio_file_path, 'rb') as audio_file:
    transcription = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_file,
    response_format="text"
)

### summary하기

In [7]:
abstract_summary = text_extraction(transcription, summary_prompt)
key_points = text_extraction(transcription, key_points_prompt)
action_items = text_extraction(transcription, action_items_prompt)
sentiment = text_extraction(transcription, sentiment_prompt)
news_data = {
    'abstract_summary': abstract_summary,
    'key_points': key_points,
    'action_items': action_items,
    'sentiment': sentiment
}

In [17]:
print(*news_data.values(), sep='\n')

짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하면서 소방대원들의 출동이 빈번해졌습니다. 말벌집 제거 요청이 급증하여 지난달에는 전년 대비 57% 증가한 44,000건의 출동이 있었습니다. 소방대원들은 특수 보호복을 착용하고 스프레이를 사용하여 벌집을 제거하며, 주민들에게는 말벌이 나타날 경우 주변을 잘 살펴보고 벌집을 신고하며, 벌 공격 시 신속히 현장을 벗어날 것을 조언하고 있습니다.
1. 짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하였습니다.
2. 소방대원들이 말벌집 제거 작업으로 바쁜 상황입니다.
3. 말벌집 제거 출동 건수가 지난해 대비 57% 증가하여 월간 44,000건에 달했습니다.
4. 소방대원들은 말벌 제거 시 특수 보호복을 착용하고 스프레이를 사용하여 벌집을 제거합니다.
5. 말벌은 어두운 색을 공격하는 경향이 있어 소방대원들은 하얀색 보호복을 착용합니다.
6. 소방당국은 말벌이 자주 나타나는 지역에서는 주변 벌집을 잘 살펴보고 신고할 것을 권장합니다.
7. 말벌 공격 시에는 현장에서 약 20미터 이상 빨리 벗어날 것을 조언합니다.
1. 주민들은 말벌이 자주 나타나는 경우 주변에 벌집이 있는지 확인하고 신고할 것.
2. 말벌 공격 시 현장에서 약 20미터 정도 빠르게 벗어날 것.
이 텍스트의 전체적인 감정은 주로 중립적이며 경고적인 요소가 포함되어 있습니다. 텍스트는 말벌의 증가와 이로 인한 소방대원의 활동 증가를 설명하고 있으며, 이는 자연 현상과 그에 따른 인간의 대응을 보여주는 보도 스타일입니다. 긍정적이거나 부정적인 감정보다는 정보 전달과 주의 권고에 초점을 맞추고 있습니다.

텍스트에서 사용된 언어는 사실적이고 설명적인 톤을 유지하고 있으며, 말벌의 위험성과 소방대원의 대응 방법을 강조하고 있습니다. 이는 청중에게 경각심을 일깨우고 안전 조치를 취하도록 권장하는 목적을 가지고 있습니다. 따라서 감정의 표현보다는 정보의 전달과 안전에 대한 조언이 주를 이루고 있습니다. 

결론적으로, 이 텍스트는 감정적으로 중립적이며, 주로 정보 

> **print(news_data)의 결과값<br/>**
{'abstract_summary': '짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하면서 소방대원들의 출동이 빈번해졌습니다. 말벌집 제거 요청은 지난해 대비 57% 증가한 44,000건에 달했습니다. 소방대원들은 보호복을 착용하고 스프레이를 사용하여 벌집을 제거하며, 말벌의 공격을 피하기 위해 주변을 잘 살피고 신속히 대피할 것을 권장합니다. 말벌은 특히 어두운 색을 공격하는 경향이 있어, 소방대원들은 하얀색 보호복을 착용합니다. 말벌의 활동이 가장 왕성한 8월과 9월에는 쏘임 사고도 가장 많이 발생합니다.', 'key_points': '1. 짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하였습니다.\n2. 소방대원들이 말벌 제거 작업에 더 자주 출동하고 있으며, 일일 최대 15건까지 출동하는 경우도 있습니다.\n3. 말벌 제거 출동 건수는 지난해 대비 57% 증가하여 한 달에 44,000건에 달했습니다.\n4. 소방대원들은 말벌 제거 시 특수 보호복을 착용하고, 벌집을 스프레이로 처리합니다.\n5. 소방당국은 말벌이 자주 나타나는 지역에서는 주변을 잘 살펴 벌집을 신고하고, 말벌 공격 시 빠르게 현장을 벗어날 것을 조언합니다.', 'action_items': '1. 소방대원들은 말벌집 제거 작업을 계속 수행합니다.\n2. 시민들은 말벌이 자주 나타나는 경우 주변에 벌집이 있는지 확인하고 신고해야 합니다.\n3. 시민들은 말벌이 공격할 때 현장에서 20미터 정도 빨리 벗어나야 합니다.', 'sentiment': '이 텍스트의 전체적인 감정은 주로 중립적이며 경고적인 요소가 포함되어 있습니다. 텍스트는 말벌의 증가와 이로 인한 소방대원의 활동 증가를 설명하고 있으며, 이는 자연 현상과 그에 따른 사회적 대응을 보고하는 뉴스 보도의 특성을 반영합니다. 글에서는 말벌로 인한 위험과 소방대원들의 대응 과정을 상세히 설명하고 있어, 독자에게 정보를 제공하고 주의를 촉구하는 목적이 강합니다.\n\n감정적으로는 긴장감이나 우려의 뉘앙스가 감지되지만, 전반적으로 객관적이고 사실적인 정보 전달에 초점을 맞추고 있습니다. 따라서 감정의 극단적인 표현보다는 사태의 심각성을 인지하고 적절히 대처하라는 조언이 주를 이루고 있습니다. 이러한 맥락에서 볼 때, 텍스트는 부정적인 상황을 보고하고 있지만, 그 처리 방식은 매우 실용적이고 해결 지향적입니다.'}

-  print(*news_data.values(), sep='\n')의 결과값<br/>

짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하면서 소방대원들의 출동이 빈번해졌습니다. 말벌집 제거 요청이 급증하여 지난달에는 전년 대비 57% 증가한 44,000건의 출동이 있었습니다. 소방대원들은 특수 보호복을 착용하고 스프레이를 사용하여 벌집을 제거하며, 주민들에게는 말벌이 나타날 경우 주변을 잘 살펴보고 벌집을 신고하며, 벌 공격 시 신속히 현장을 벗어날 것을 조언하고 있습니다.
1. 짧은 장마와 긴 폭염으로 인해 말벌 활동이 증가하였습니다.
2. 소방대원들이 말벌집 제거 작업으로 바쁜 상황입니다.
3. 말벌집 제거 출동 건수가 지난해 대비 57% 증가하여 월간 44,000건에 달했습니다.
4. 소방대원들은 말벌 제거 시 특수 보호복을 착용하고 스프레이를 사용하여 벌집을 제거합니다.
5. 말벌은 어두운 색을 공격하는 경향이 있어 소방대원들은 하얀색 보호복을 착용합니다.
6. 소방당국은 말벌이 자주 나타나는 지역에서는 주변 벌집을 잘 살펴보고 신고할 것을 권장합니다.
7. 말벌 공격 시에는 현장에서 약 20미터 이상 빨리 벗어날 것을 조언합니다.
1. 주민들은 말벌이 자주 나타나는 경우 주변에 벌집이 있는지 확인하고 신고할 것.
2. 말벌 공격 시 현장에서 약 20미터 정도 빠르게 벗어날 것.
이 텍스트의 전체적인 감정은 주로 중립적이며 경고적인 요소가 포함되어 있습니다. 텍스트는 말벌의 증가와 이로 인한 소방대원의 활동 증가를 설명하고 있으며, 이는 자연 현상과 그에 따른 인간의 대응을 보여주는 보도 스타일입니다. 긍정적이거나 부정적인 감정보다는 정보 전달과 주의 권고에 초점을 맞추고 있습니다.

텍스트에서 사용된 언어는 사실적이고 설명적인 톤을 유지하고 있으며, 말벌의 위험성과 소방대원의 대응 방법을 강조하고 있습니다. 이는 청중에게 경각심을 일깨우고 안전 조치를 취하도록 권장하는 목적을 가지고 있습니다. 따라서 감정의 표현보다는 정보의 전달과 안전에 대한 조언이 주를 이루고 있습니다. 

결론적으로, 이 텍스트는 감정적으로 중립적이며, 주로 정보 제공과 안전 조치에 대한 권고에 중점을 두고 있습니다.